# Week 59, Day 3: Leakage: when test data sneak into training

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w059/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Selection outside and inside the folds

Write top_features(X, y, k), returning the k features with the largest absolute difference in class means. Then run 5-fold cross-validation of a nearest-centroid decoder twice: once with features chosen on all 40 trials, once with features chosen on the training folds only.

In [ ]:
import numpy as np
gl = np.random.default_rng(593)
Xn = gl.normal(size=(40, 2000))
yn = np.tile([0, 1], 20)

def top_features(X, y, k=10):
    diff = ...
    return np.argsort(diff)[-k:]

def centroid_cv(X, y, select_inside):
    folds = np.array_split(np.random.default_rng(0).permutation(len(y)), 5)
    fixed = None if select_inside else top_features(X, y)
    right = 0
    for te in folds:
        tr = np.setdiff1d(np.arange(len(y)), te)
        f = top_features(X[tr], y[tr]) if select_inside else fixed
        c0, c1 = X[tr][y[tr] == 0][:, f].mean(0), X[tr][y[tr] == 1][:, f].mean(0)
        d0 = ((X[te][:, f] - c0) ** 2).sum(1); d1 = ((X[te][:, f] - c1) ** 2).sum(1)
        right += np.sum((d1 < d0).astype(int) == y[te])
    return right / len(y)

leaky, honest = centroid_cv(Xn, yn, False), centroid_cv(Xn, yn, True)
print(f"select on all data, then CV: {leaky:.3f}")
print(f"select inside each fold:     {honest:.3f}")

In [ ]:
import numpy as np
def test_leak():
    "leaky selection reports accuracy above 0.75 on pure noise"
    assert leaky > 0.75
def test_honest():
    "selection inside each training fold gives accuracy near chance"
    assert 0.3 < honest < 0.7

test_leak()
test_honest()
print("All checks passed")

### Exercise: Drift and a blocked design

Simulate 400 trials in blocks of 25, alternating between two conditions, from 20 neurons whose rates drift as random walks and carry no information about the condition. Score a 1-nearest-neighbour decoder with 5 random folds and with 5 folds made of whole consecutive blocks.

In [ ]:
gt = np.random.default_rng(5931)
T, B = 400, 25
cond = (np.arange(T) // B) % 2                       # blocks of 25 trials, alternating conditions
drift = np.cumsum(0.3 * gt.normal(size=(T, 20)), axis=0)
Xd = drift + gt.normal(size=(T, 20))                 # no condition signal at all

def nn_cv(X, y, folds):
    right = 0
    for te in folds:
        tr = np.setdiff1d(np.arange(len(y)), te)
        d = ((X[te][:, None, :] - X[tr][None, :, :]) ** 2).sum(-1)
        right += np.sum(y[tr][np.argmin(d, axis=1)] == y[te])
    return right / len(y)

random_folds = np.array_split(gt.permutation(T), 5)
blocked_folds = ...
acc_random, acc_blocked = nn_cv(Xd, cond, random_folds), nn_cv(Xd, cond, blocked_folds)
print(f"random folds: {acc_random:.3f}; blocked folds: {acc_blocked:.3f}")

In [ ]:
import numpy as np
def test_random():
    "randomly split cross-validation decodes the block condition from drift alone"
    assert acc_random > 0.75
def test_blocked():
    "folds made of whole blocks bring accuracy back near chance"
    assert acc_blocked < 0.65

test_random()
test_blocked()
print("All checks passed")

### Exercise: Leave one session out

Six sessions each have their own recording offset across 30 neurons, and in each session most trials happen to be of one choice, a different one in alternate sessions; the neurons carry no choice signal. Compare 6-fold cross-validation over shuffled trials with leave-one-session-out, using the nearest-neighbour decoder from Step 3.

In [ ]:
gs = np.random.default_rng(5932)
sessions, labels, rows = [], [], []
for s in range(6):
    offset = 2 * gs.normal(size=30)                       # session-specific recording offset
    p_right = 0.9 if s % 2 == 0 else 0.1                  # the animal's bias differs by session
    ys = (gs.random(60) < p_right).astype(int)
    rows.append(offset + gs.normal(size=(60, 30))); labels.append(ys); sessions.append(np.full(60, s))
Xs, ys_all, sess = np.vstack(rows), np.concatenate(labels), np.concatenate(sessions)

trial_folds = np.array_split(gs.permutation(len(ys_all)), 6)
session_folds = ...
acc_trial, acc_session = nn_cv(Xs, ys_all, trial_folds), nn_cv(Xs, ys_all, session_folds)
print(f"split by trial: {acc_trial:.3f}; leave one session out: {acc_session:.3f}")

In [ ]:
import numpy as np
def test_trialwise():
    "splitting by trial reaches accuracy above 0.7"
    assert acc_trial > 0.7
def test_group():
    "leaving whole sessions out reveals chance-level generalisation"
    assert acc_session < 0.65

test_trialwise()
test_group()
print("All checks passed")